# Cruzamento reproduzível — PIB x PAM x PPM

## Chave, cardinalidade e correspondência

Este notebook faz só o cruzamento das 3 bases obrigatórias (PIB, PAM, PPM) no grão município x ano, e documenta correspondências/não-correspondências. Insights, visualizações e a transformação cruzada ficam em `integracao.ipynb`, que consome o resultado salvo aqui em `dados/processed/`.

In [1]:
import pandas as pd
import numpy as np

## 1. Carga e preparação — grão comum (município × ano)

**PIB** — já está no grão município × ano por natureza (uma linha por variável). Pivota as 4 variáveis para colunas.

In [2]:
pib = pd.read_csv(
    "../dados/raw/dados_originais/t5938_pib_agropecuaria_2003_2023_ce_br.csv",
    sep=";", encoding="utf-8-sig"
)
pib["valor"] = pd.to_numeric(pib["valor"], errors="coerce")
pib_mun = pib[pib["nivel_territorial_nome"] == "Município"].copy()
pib_mun["territorio_nome"] = pib_mun["territorio_nome"].str.replace(r" - CE$", "", regex=True)

pib_wide = pib_mun.pivot_table(
    index=["territorio_codigo", "ano_codigo"], columns="variavel_nome", values="valor"
).reset_index()

pib_wide = pib_wide.rename(columns={
    "Produto Interno Bruto a preços correntes": "PIB",
    "Valor adicionado bruto a preços correntes total": "VAB_total",
    "Valor adicionado bruto a preços correntes da agropecuária": "VAB_agro",
    "Participação do valor adicionado bruto a preços correntes da agropecuária no valor adicionado bruto a preços correntes total": "participacao_agro",
})

pib_wide.shape
pib_wide.head()

variavel_nome,territorio_codigo,ano_codigo,participacao_agro,PIB,VAB_agro,VAB_total
0,2300101,2003,34.46,17657.0,5897.0,17110.0
1,2300101,2004,26.28,16808.0,4235.0,16118.0
2,2300101,2005,18.69,17153.0,3059.0,16367.0
3,2300101,2006,23.04,22869.0,5060.0,21960.0
4,2300101,2007,19.74,25383.0,4831.0,24475.0


**PAM** — soma o valor da produção (Mil Reais) dos 7 produtos por município × ano. Somar é válido aqui porque é valor monetário (mesma unidade); diferente de somar quantidade produzida entre produtos com unidades distintas, o que o projeto explicitamente veda.

In [3]:
pam_paths = [
    "../dados/raw/dados_originais/t5457_pam_area_colhida_2003_2024_ce_br.csv",
    "../dados/raw/dados_originais/t5457_pam_area_plantada_2003_2024_ce_br.csv",
    "../dados/raw/dados_originais/t5457_pam_quantidade_produzida_2003_2024_ce_br.csv",
    "../dados/raw/dados_originais/t5457_pam_rendimento_medio_2003_2024_ce_br.csv",
    "../dados/raw/dados_originais/t5457_pam_valor_producao_2003_2024_ce_br.csv",
]

pam = pd.concat(
    [pd.read_csv(p, sep=";", encoding="utf-8-sig") for p in pam_paths],
    ignore_index=True
)
pam["valor"] = pd.to_numeric(pam["valor"], errors="coerce")
is_rendimento = pam["variavel_nome"] == "Rendimento médio da produção"
pam.loc[~is_rendimento, "valor"] = pam.loc[~is_rendimento, "valor"].fillna(0)

pam_mun = pam[pam["nivel_territorial_nome"] == "Município"].copy()

pam_valor_producao = (
    pam_mun[pam_mun["variavel_nome"] == "Valor da produção"]
    .groupby(["territorio_codigo", "ano_codigo"])["valor"].sum()
    .reset_index(name="valor_producao_agricola_total")
)

pam_valor_producao.shape
pam_valor_producao.head()

,territorio_codigo,ano_codigo,valor_producao_agricola_total
0,2300101,2003,3093.0
1,2300101,2004,1775.0
2,2300101,2005,689.0
3,2300101,2006,1686.0
4,2300101,2007,2059.0


**PPM** — usa só o efetivo bovino por município × ano. Não soma as espécies (bovino + caprino + ovino + suíno + galináceos) — o projeto veda tratar efetivos de espécies distintas como equivalentes.

In [4]:
ppm = pd.read_csv(
    "../dados/raw/dados_originais/t3939_ppm_efetivo_rebanhos_2003_2024_ce_br.csv",
    sep=";", encoding="utf-8-sig"
)
ppm["valor"] = pd.to_numeric(ppm["valor"], errors="coerce").fillna(0)
ppm_mun = ppm[ppm["nivel_territorial_nome"] == "Município"].copy()

ppm_bovino = (
    ppm_mun[ppm_mun["tipo_rebanho_nome"] == "Bovino"]
    [["territorio_codigo", "ano_codigo", "valor"]]
    .rename(columns={"valor": "efetivo_bovino"})
)

ppm_bovino.shape
ppm_bovino.head()

,territorio_codigo,ano_codigo,efetivo_bovino
44,2300101,2003,5311.0
45,2300101,2004,5433.0
46,2300101,2005,5615.0
47,2300101,2006,5790.0
48,2300101,2007,5969.0


## 2. Alinhamento temporal

PIB municipal cobre 2003-2023 (e o VAB derivado só está completo até 2021 — ver `PIB.ipynb`). PAM e PPM cobrem 2003-2024. Isso já avisa que o cruzamento vai ter linhas "só PAM/PPM" nos anos 2022-2024, não é erro de chave.

In [5]:
pd.DataFrame({
    "base": ["PIB", "PAM", "PPM"],
    "ano_min": [pib_wide["ano_codigo"].min(), pam_valor_producao["ano_codigo"].min(), ppm_bovino["ano_codigo"].min()],
    "ano_max": [pib_wide["ano_codigo"].max(), pam_valor_producao["ano_codigo"].max(), ppm_bovino["ano_codigo"].max()],
})

,base,ano_min,ano_max
0,PIB,2003,2023
1,PAM,2003,2024
2,PPM,2003,2024


## 3. Cruzamento reproduzível

**Chave:** `territorio_codigo` + `ano_codigo`.
**Cardinalidade esperada:** 1:1 em cada base (já agregamos pra 1 linha por município por ano) — `validate="one_to_one"` interrompe o merge se isso for violado.
**Universo máximo possível:** 184 municípios × anos da união (2003-2024 = 22 anos) = 4048 combinações, se as 3 bases cobrissem os mesmos anos — não cobrem (ver seção 2), então não esperamos bater 100%.

In [6]:
merge_pib_pam = pib_wide.merge(
    pam_valor_producao,
    on=["territorio_codigo", "ano_codigo"],
    how="outer",
    validate="one_to_one",
    indicator="_merge_pib_pam",
)

merge_pib_pam["_merge_pib_pam"].value_counts()

_merge_pib_pam
both          3864
right_only     184
left_only        0
Name: count, dtype: int64

`both` = município×ano presente nas duas bases. `left_only` = só no PIB (esperado: nenhum, já que PAM cobre todo o período do PIB). `right_only` = só no PAM (esperado: os anos 2022-2024, que o PIB não tem).

In [7]:
merge_final = merge_pib_pam.drop(columns="_merge_pib_pam").merge(
    ppm_bovino,
    on=["territorio_codigo", "ano_codigo"],
    how="outer",
    validate="one_to_one",
    indicator="_merge_ppm",
)

merge_final["_merge_ppm"].value_counts()

_merge_ppm
both          4048
left_only        0
right_only       0
Name: count, dtype: int64

## 4. Correspondência final — presença por base

Usa a coluna `PIB` (sempre presente 2003-2023, não tem lacuna) e `valor_producao_agricola_total` / `efetivo_bovino` como indicador de "essa linha veio dessa base", já que essas colunas só ficam `NaN` quando o `outer merge` não encontrou a chave na base de origem.

In [8]:
merge_final["em_pib"] = merge_final["PIB"].notna()
merge_final["em_pam"] = merge_final["valor_producao_agricola_total"].notna()
merge_final["em_ppm"] = merge_final["efetivo_bovino"].notna()

resumo_correspondencia = pd.DataFrame({
    "base": ["PIB", "PAM", "PPM"],
    "linhas na base original (município×ano)": [
        len(pib_wide), len(pam_valor_producao), len(ppm_bovino)
    ],
    "linhas no cruzamento final": [
        merge_final["em_pib"].sum(), merge_final["em_pam"].sum(), merge_final["em_ppm"].sum()
    ],
    "presentes nas 3 bases (%)": [
        (merge_final["em_pib"] & merge_final["em_pam"] & merge_final["em_ppm"]).sum() / merge_final["em_pib"].sum() * 100,
        (merge_final["em_pib"] & merge_final["em_pam"] & merge_final["em_ppm"]).sum() / merge_final["em_pam"].sum() * 100,
        (merge_final["em_pib"] & merge_final["em_pam"] & merge_final["em_ppm"]).sum() / merge_final["em_ppm"].sum() * 100,
    ],
})

resumo_correspondencia.round(1)

,base,linhas na base original (município×ano),linhas no cruzamento final,presentes nas 3 bases (%)
0,PIB,3864,3864,100.0
1,PAM,4048,4048,95.5
2,PPM,4048,4048,95.5


In [9]:
# quantas linhas do cruzamento NÃO têm as 3 bases ao mesmo tempo, e em que anos isso concentra
sem_as_tres = merge_final[~(merge_final["em_pib"] & merge_final["em_pam"] & merge_final["em_ppm"])]
len(sem_as_tres)
sem_as_tres["ano_codigo"].value_counts().sort_index()

ano_codigo
2024    184
Name: count, dtype: int64

## 5. Salvar para consumo em `integracao.ipynb`

Salva o cruzamento completo (com nome do município) e a fatia da PAM por produto (necessária pra análise de produto dominante) em `dados/processed/`, pra `integracao.ipynb` não precisar reprocessar as bases raw.

In [10]:
codigo_nome = pib_mun[["territorio_codigo", "territorio_nome"]].drop_duplicates()
merge_final_nomeado = merge_final.merge(codigo_nome, on="territorio_codigo", how="left")

merge_final_nomeado.to_parquet("../dados/processed/cruzamento_pib_pam_ppm.parquet", index=False)

valor_producao_mun = pam_mun[pam_mun["variavel_nome"] == "Valor da produção"][
    ["territorio_codigo", "ano_codigo", "produto_nome", "valor"]
]
valor_producao_mun.to_parquet("../dados/processed/pam_valor_producao_produto.parquet", index=False)

merge_final_nomeado.shape, valor_producao_mun.shape

((4048, 13), (28336, 4))